# Critical X: X21 / 90° 회전 Y17 비교
이 파일을 `C:\PHOTO_ANALYSIS\BASIC\CPE_Critical_Rotation.ipynb`에 놓고 **00부터 순서대로 실행**합니다. 기존 `stages`의 Parquet/전체 Excel만 읽으며 DB 조회나 원본 노트북 실행은 필요하지 않습니다.

문제 제품의 **같은 X 오차·같은 측정 point**에 두 모델을 적용합니다. 입력은 `21_ADI_WK_RK_REMOVED` / `21_OCO_WK_RK_REMOVED`의 `raw_minus_total_fit_x`입니다. 21 파일이 없으면 같은 입력 컬럼이 있는 27 FINAL_POINTS를 사용합니다. **WK/RK는 다시 계산하지 않습니다.**

- **X21_NATIVE:** 기존 장비 좌표의 X21 모델과 X측 ridge/G-opt/span 설정.
- **ROT_Y17_NATIVE:** 로컬 좌표와 critical X 성분을 ±90° 회전 → Y17과 Y측 설정으로 회귀 → 보정값을 원래 장비 X로 복원.
- **OLS_DIAGNOSTIC:** 두 항 구성의 제약 없는 회귀도 별도 저장합니다. 실제 보정 설정 결과와 구분해서 보세요.

**원래 좌표에서 X21과 회전 Y17은 포함관계가 아닙니다.** 공통 단항식은 12개이고, X21만 있는 항은 9개, 회전 Y17만 있는 항은 5개입니다. 예를 들어 원래 좌표의 `x^4`는 X21에 없고 회전 Y17에는 있습니다. 07에서 전체 대응표를 저장합니다.

00의 `FILTERS`에 문제 제품의 LOT/recipe 등을 지정하세요. 필요하면 앞서 저장한 `output_wafer_context\wafer_context.parquet`도 자동 연결합니다. **필터가 비어 있으면 선택한 mode의 전체 입력을 critical X 제품으로 취급합니다.** Y 측정값은 사용하지 않습니다.

원단위는 µm, 통계 출력은 nm입니다. 결과는 새 `output_cpe_critical_rotation`에 저장합니다. 제거된 CPE가 작다는 것만으로 개선을 판단하지 말고 **최종 residual의 3σ 및 |mean|+3σ**를 함께 보세요. 학습에 사용한 point의 회귀 결과이며 장비에 실제 보정을 적용한 결과는 아닙니다.


## 00. 경로 / 대상 제품 / 비교 설정


In [ ]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import json
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

BASE_DIR = Path(r"C:\PHOTO_ANALYSIS\BASIC")
STAGE_DIR = BASE_DIR / "output_decorrect_notebook" / "stages"
OUTPUT_DIR = BASE_DIR / "output_cpe_critical_rotation"
MODES = ["ADI", "OCO"]
FILTERS = {}  # 예: {"LOT_ID": ["LOT_A", "LOT_B"]}; 여러 컬럼은 AND, 컬럼 안의 값은 OR
FILTERS_BY_MODE = {"ADI": {}, "OCO": {}}  # 필요하면 mode별 제품/recipe 필터 추가
ROTATION_DEG = 90  # +90 또는 -90; 원래 장비 X 오차를 회전 좌표의 Y 오차로 변환
USE_SAVED_SOLVER_SETTINGS = True
SOLVER_OVERRIDES = {}  # 예: {"G_OPT_SPEC": 1.0}; 아래 원본 solver 셀 실행 후 적용
RUN_OLS_DIAGNOSTIC = True  # ridge/span 없는 진단 회귀; 실측 point에서의 설명력만 비교
USE_WAFER_CONTEXT = True
STD_DDOF = 1
UM_TO_NM = 1000.0
EXCEL_PREVIEW_ROWS = 10000
SHOW_PLOTS = True
PREVIEW_ROWS = 8
WAFER_KEYS = ["LOT_ID", "Wafer", "photo_transn_seq"]
SHOT_KEYS = WAFER_KEYS + ["fcp_x", "fcp_y"]
COORDS = ["coordinate_X", "coordinate_Y"]
TARGET = "raw_minus_total_fit_x"
NATIVE_CASES = ["X21_NATIVE", "ROT_Y17_NATIVE"]
if not MODES or any(mode not in ["ADI", "OCO"] for mode in MODES) or len(set(MODES)) != len(MODES):
    raise ValueError("MODES는 ADI/OCO 중 중복 없이 지정하세요.")
if ROTATION_DEG not in [90, -90]:
    raise ValueError("ROTATION_DEG는 +90 또는 -90입니다.")
if not STAGE_DIR.is_dir():
    raise FileNotFoundError(f"stages 폴더 없음: {STAGE_DIR}")
TABLE_DIR, PLOT_DIR = OUTPUT_DIR / "tables", OUTPUT_DIR / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
input_manifest, output_manifest = [], {}
plt.rcParams.update({"axes.unicode_minus": False, "figure.dpi": 110})
print("입력:", STAGE_DIR)
print("출력:", OUTPUT_DIR)
print("선택된 모든 wafer는 critical X 제품으로 취급합니다. 제품 필터는 FILTERS에 지정하세요.")


## 01. 저장 파일 읽기 / ID 정규화 / 중간 결과 저장


In [ ]:
def normalized_id(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    try:
        number = Decimal(value)
        return str(int(number)) if number.is_finite() and number == number.to_integral_value() else value
    except (InvalidOperation, ValueError, OverflowError):
        return value

def normalize_keys(frame):
    out = frame.copy()
    for key in WAFER_KEYS:
        if key not in out:
            raise ValueError(f"wafer 키 누락: {key}")
        values = out[key].astype("string").str.strip()
        if key != "LOT_ID":
            lookup = {value: normalized_id(value) for value in values.dropna().unique()}
            values = values.map(lookup)
        out[key] = values.astype("string")
    return out

def read_full(stems, directory=STAGE_DIR, required=True):
    failures = []
    for stem in stems:
        for suffix in [".parquet", ".xlsx"]:
            path = directory / (stem + suffix)
            if not path.is_file():
                continue
            try:
                frame = pd.read_parquet(path) if suffix == ".parquet" else pd.read_excel(path, dtype={key: "string" for key in WAFER_KEYS})
            except Exception as exc:
                failures.append(f"{path.name}: {type(exc).__name__}: {exc}")
                continue
            input_manifest.append(dict(path=str(path), rows=len(frame), columns=len(frame.columns), bytes=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns, fallback_errors="; ".join(failures)))
            return frame
    if required:
        raise FileNotFoundError(f"전체 Parquet/Excel 필요: {stems}. sample Excel은 입력에서 제외합니다. {'; '.join(failures)}")
    if failures:
        print("선택 입력 읽기 실패:", "; ".join(failures))
    return None

def excel_safe(frame):
    out = frame.copy()
    for col in out.select_dtypes(include=["object", "string"]).columns:
        out[col] = out[col].map(lambda value: "'" + value if isinstance(value, str) and value.startswith(("=", "+", "-", "@")) else value)
    return out

def save_table(name, frame, excel=True):
    frame.to_parquet(TABLE_DIR / f"{name}.parquet", index=False)
    frame.to_csv(TABLE_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    excel_path = None
    if excel:
        sample = len(frame) > EXCEL_PREVIEW_ROWS
        excel_path = TABLE_DIR / (f"{name}_sample_{EXCEL_PREVIEW_ROWS}.xlsx" if sample else f"{name}.xlsx")
        excel_safe(frame.head(EXCEL_PREVIEW_ROWS)).to_excel(excel_path, index=False, engine="openpyxl")
    output_manifest[name] = dict(name=name, rows=len(frame), columns=len(frame.columns), parquet=str(TABLE_DIR / f"{name}.parquet"), excel=str(excel_path) if excel_path else None)
    print(f"[{name}] {len(frame):,} rows × {len(frame.columns)} columns")
    if excel_path is not None:
        display(FileLink(os.path.relpath(excel_path, Path.cwd())))

def apply_filters(frame, mode):
    out = frame
    for rules in [FILTERS, FILTERS_BY_MODE.get(mode, {})]:
        for col, values in rules.items():
            if col not in out:
                raise ValueError(f"{mode}: 필터 컬럼 없음: {col}. 02_input_columns.csv를 확인하세요.")
            if not isinstance(values, (list, tuple, set)) or not values:
                raise ValueError(f"{col}: 필터값은 비어 있지 않은 list/tuple/set으로 지정하세요.")
            wanted = [str(normalized_id(value)) if col in WAFER_KEYS[1:] else str(value).strip() for value in values]
            out = out.loc[out[col].astype("string").str.strip().isin(wanted)]
    return out.reset_index(drop=True)


## 02. 대상 wafer 선택 / CPE 직전 X 데이터 확인


In [ ]:
frames, selection_rows, column_rows = {}, [], []
context = read_full(["wafer_context"], BASE_DIR / "output_wafer_context", required=False) if USE_WAFER_CONTEXT else None
if context is not None:
    context = normalize_keys(context)
    if context[WAFER_KEYS].isna().any(axis=None) or context.duplicated(WAFER_KEYS).any():
        raise ValueError("wafer_context는 전체 wafer 키가 유효하고 유일해야 합니다.")
for mode in MODES:
    frame = normalize_keys(read_full([f"21_{mode}_WK_RK_REMOVED", f"27_{mode}_FINAL_POINTS"]))
    if context is not None:
        frame = frame.merge(context, how="left", on=WAFER_KEYS, validate="many_to_one", suffixes=("", "_context"))
    column_rows.extend(dict(mode=mode, column=col, dtype=str(frame[col].dtype)) for col in frame)
    pd.DataFrame(column_rows).to_csv(TABLE_DIR / "02_input_columns.csv", index=False, encoding="utf-8-sig")
    frame = apply_filters(frame, mode)
    missing = [col for col in SHOT_KEYS + COORDS + [TARGET] if col not in frame]
    if missing:
        raise ValueError(f"{mode}: CPE 직전 입력 컬럼 누락: {missing}")
    if frame.empty:
        raise ValueError(f"{mode}: 제품 필터에 맞는 데이터가 없습니다.")
    if frame[WAFER_KEYS].isna().any(axis=None):
        raise ValueError(f"{mode}: wafer 키에 결측값이 있습니다.")
    for col in SHOT_KEYS[3:] + COORDS + [TARGET]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    frame["source_row_id"] = np.arange(len(frame), dtype=np.int64)
    frame["rotation_valid_x"] = np.isfinite(frame[SHOT_KEYS[3:] + COORDS + [TARGET]].to_numpy(float)).all(axis=1)
    frames[mode] = frame
    selected = frame.groupby(WAFER_KEYS, dropna=False, sort=False).agg(n_points=(TARGET, "size"), n_valid_x=("rotation_valid_x", "sum")).reset_index()
    selected.insert(0, "mode", mode)
    selection_rows.append(selected)
    print(mode, "wafer 수:", len(selected), "유효 X point:", int(frame["rotation_valid_x"].sum()))
selection = pd.concat(selection_rows, ignore_index=True)
save_table("02_SELECTED_WAFERS", selection)
save_table("02_INPUT_MANIFEST", pd.DataFrame(input_manifest))
display(selection.head(PREVIEW_ROWS))


## 03. CPE38 설정, 행렬 및 span 제한
원본 MRC_PSM_decorrect의 22번 정의를 그대로 포함합니다. 실제 설정은 07에서 저장 설정 및 00의 override를 반영합니다.


In [ ]:
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

SPAN_MODE = "OVO3_38P_F2F"
RIDGE_REGRESSION = True
G_OPT_SPEC = 1.0
USE_VIRTUAL_CHIP_POINTS = True   # VBA virtual grid: chip별 4 corner + center point
G_OPT_TOLERANCE = 0.001          # VBA: G_opt_tolerlance = 0.001
SPAN_TOLERANCE = 0.01            # VBA: span_tolerlance = 0.01
MINIMUM_RIDGE = 0.001            # VBA: minimum_Ridge = 0.001
ADD_RIDGE = 10.0                 # VBA: add_ridge = 10
MAX_GOPT_RETURN = 50             # VBA: max_prevent_fallback_retrun = 50
MAX_SPAN_ITER = 5000             # VBA: max_iter = 5000
CD_TOL = 1e-6                    # VBA: tol = 0.000001
CPE_EVAL_GRID_N = 10
MIN_POINTS_GUIDE = 10
G_OPTIMAL_TYPE = 1               # VBA information 기준: GOT1
VIRTUAL_WAFER_RADIUS = 144000.0  # VBA virtual point keep 조건: R <= 144000               # VBA option D59: 1,2,3,4. 4 = inner OT2 / partial OT1
USE_SPAN_BOX_CONSTRAINT = True   # VBA Balloon projected CD/PGD stage

def cpe_38para(rx: Sequence[float], ry: Sequence[float]) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27,
        (rx ** 3 * ry ** 4) / 1e31,
    ]).T
    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23,
        (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27,
    ]).T
    return X_dx, X_dy
CPE38_KEYS_DX = [
    "RK1", "RK3", "RK5",
    "RK7", "RK9", "RK11",
    "RK13", "RK15", "RK17", "RK19",
    "RK23", "RK25", "RK27", "RK29",
    "RK35", "RK37", "RK39", "RK41",
    "RK49", "RK51",
    "RK65",
]
CPE38_KEYS_DY = [
    "RK2", "RK4", "RK6",
    "RK8", "RK10", "RK12",
    "RK14", "RK16", "RK18",
    "RK22", "RK24", "RK26",
    "RK32", "RK34", "RK36",
    "RK46", "RK48",
]
RK_SPAN_LIMITS_OVO3_38P_F2F = {
    'RK1': {'min': -0.1, 'max': 0.1},
    'RK2': {'min': -0.1, 'max': 0.1},
    'RK3': {'min': -2.0, 'max': 2.0},
    'RK4': {'min': -10.0, 'max': 10.0},
    'RK5': {'min': -5.0, 'max': 5.0},
    'RK6': {'min': -5.0, 'max': 5.0},
    'RK7': {'min': -0.1, 'max': 0.1},
    'RK8': {'min': -0.15, 'max': 0.15},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -0.5, 'max': 0.5},
    'RK11': {'min': -0.5, 'max': 0.5},
    'RK12': {'min': -0.03, 'max': 0.03},
    'RK13': {'min': -0.003, 'max': 0.003},
    'RK14': {'min': -0.015, 'max': 0.015},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -0.05, 'max': 0.05},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -0.023, 'max': 0.023},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}
RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT = {
    'RK1': {'min': -100.0, 'max': 100.0},
    'RK2': {'min': -100.0, 'max': 100.0},
    'RK3': {'min': -1000.0, 'max': 1000.0},
    'RK4': {'min': -1000.0, 'max': 1000.0},
    'RK5': {'min': -1000.0, 'max': 1000.0},
    'RK6': {'min': -1000.0, 'max': 1000.0},
    'RK7': {'min': -0.25, 'max': 0.25},
    'RK8': {'min': -10.0, 'max': 10.0},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -10.0, 'max': 10.0},
    'RK11': {'min': -10.0, 'max': 10.0},
    'RK12': {'min': -0.25, 'max': 0.25},
    'RK13': {'min': -0.025, 'max': 0.025},
    'RK14': {'min': -1.0, 'max': 1.0},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -1.0, 'max': 1.0},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -1.0, 'max': 1.0},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}

def get_active_span_limits(span_mode: str = "OVO3_38P_F2F") -> Dict[str, Dict[str, float]]:
    if span_mode == "OVO3_38P_HARDLIMIT":
        return RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT
    return RK_SPAN_LIMITS_OVO3_38P_F2F
RIDGE_VALVE_HYPER = {
    "RK1": 0.0, "RK2": 0.0,
    "RK3": 0.01, "RK4": 0.01, "RK5": 0.01, "RK6": 0.01,
    "RK7": 1.0, "RK8": 0.1, "RK9": 1.0, "RK10": 0.1,
    "RK11": 0.1, "RK12": 1.0,
    "RK13": 1.0, "RK14": 0.1, "RK15": 1.0, "RK16": 0.1,
    "RK17": 1.0, "RK18": 1.0, "RK19": 0.1, "RK20": 1.0,
}


## 04. CPE virtual grid 및 소표본 ridge 배율
원본 MRC_PSM_decorrect의 23번 정의를 그대로 포함합니다. 실제 설정은 07에서 저장 설정 및 00의 override를 반영합니다.


In [ ]:
def _to_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def choose_base_lambda_scale(n_points: int) -> float:
    """VBA의 point가 적어도 skip하지 않는 방식을 Python식으로 반영."""
    if n_points >= 10:
        return 1.0
    if n_points >= 7:
        return 1e2
    if n_points >= 5:
        return 1e4
    if n_points >= 3:
        return 1e6
    return 1e8

def get_ridge_valves(coeff_names: Sequence[str]) -> np.ndarray:
    return np.array([RIDGE_VALVE_HYPER.get(name, 1.0) for name in coeff_names], dtype=float)

def make_lambda_vector(base_lambda: float, coeff_names: Sequence[str]) -> np.ndarray:
    valves = get_ridge_valves(coeff_names)
    return float(base_lambda) * valves

def build_virtual_chip_points(
    step_pitch_x: float,
    step_pitch_y: float,
    chip_x_num: int,
    chip_y_num: int,
    round_decimals: int = 6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA virtual grid 재현:
    chip별 4 corner + center point를 shot-local coordinate로 생성하고,
    chip 경계에서 겹치는 corner는 dedup한다.
    예: chip_x_num=2, chip_y_num=1
      - 각 chip 5 point => 10 raw points
      - 가운데 공유 corner 2개 중복 제거
      - 최종 8 virtual points
    coordinate_X/Y가 shot center 기준이라고 보고,
    전체 shot 영역은 [-STEP_PITCH_X/2, +STEP_PITCH_X/2],
                  [-STEP_PITCH_Y/2, +STEP_PITCH_Y/2] 로 둔다.
    """
    chip_x_num = int(chip_x_num) if pd.notna(chip_x_num) and int(chip_x_num) > 0 else 1
    chip_y_num = int(chip_y_num) if pd.notna(chip_y_num) and int(chip_y_num) > 0 else 1
    sx = float(step_pitch_x)
    sy = float(step_pitch_y)
    x_edges = np.linspace(-sx / 2.0, sx / 2.0, chip_x_num + 1)
    y_edges = np.linspace(-sy / 2.0, sy / 2.0, chip_y_num + 1)
    pts = []
    for ix in range(chip_x_num):
        for iy in range(chip_y_num):
            x0, x1 = x_edges[ix], x_edges[ix + 1]
            y0, y1 = y_edges[iy], y_edges[iy + 1]
            xc = (x0 + x1) / 2.0
            yc = (y0 + y1) / 2.0
            pts.extend([
                (x0, y0), (x0, y1), (x1, y0), (x1, y1),
                (xc, yc),
            ])
    pts_unique = sorted({(round(x, round_decimals), round(y, round_decimals)) for x, y in pts})
    rx = np.array([p[0] for p in pts_unique], dtype=float)
    ry = np.array([p[1] for p in pts_unique], dtype=float)
    return rx, ry

def filter_virtual_points_by_wafer_radius(
    rx_v: Sequence[float],
    ry_v: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    VBA edge-shot virtual point filtering.
    virtual point의 wafer 좌표 R이 wafer_radius 이하인 point만 사용한다.
    R = sqrt((fcp_x + rx_v)^2 + (fcp_y + ry_v)^2)
    keep if R <= 144000
    """
    rx_v = np.asarray(rx_v, dtype=float)
    ry_v = np.asarray(ry_v, dtype=float)
    wx = float(fcp_x) + rx_v
    wy = float(fcp_y) + ry_v
    r = np.sqrt(wx * wx + wy * wy)
    keep = r <= float(wafer_radius)
    return rx_v[keep], ry_v[keep], r[keep]

def get_virtual_points_for_group(
    grp: pd.DataFrame,
    fallback_rx: Sequence[float],
    fallback_ry: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, str, int, int]:
    """
    1) full virtual point 생성: chip별 4 corner + center
    2) edge shot 처리: wafer 좌표 R <= wafer_radius point만 keep
    반환:
      rx_v, ry_v, virtual_source, n_virtual_full, n_virtual_kept
    """
    required = ["STEP_PITCH_X", "STEP_PITCH_Y", "CHIP_X_NUM", "CHIP_Y_NUM"]
    if all(c in grp.columns for c in required):
        vals = grp[required].dropna()
        if len(vals) > 0:
            row = vals.iloc[0]
            rx_full, ry_full = build_virtual_chip_points(
                step_pitch_x=row["STEP_PITCH_X"],
                step_pitch_y=row["STEP_PITCH_Y"],
                chip_x_num=row["CHIP_X_NUM"],
                chip_y_num=row["CHIP_Y_NUM"],
            )
            n_full = len(rx_full)
            rx_keep, ry_keep, _ = filter_virtual_points_by_wafer_radius(
                rx_full,
                ry_full,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=wafer_radius,
            )
            if len(rx_keep) > 0:
                src = "chip_corner_center_wafer_clipped"
                return rx_keep, ry_keep, src, n_full, len(rx_keep)
            return rx_full, ry_full, "chip_corner_center_no_clip_points_fallback", n_full, n_full
    rx_v, ry_v = build_eval_grid_from_data(fallback_rx, fallback_ry, grid_n=CPE_EVAL_GRID_N)
    return rx_v, ry_v, "measured_range_grid_fallback", len(rx_v), len(rx_v)


## 05. Ridge 예측 및 G-opt 평가 함수
원본 MRC_PSM_decorrect의 24번 정의를 그대로 포함합니다. 실제 설정은 07에서 저장 설정 및 00의 override를 반영합니다.


In [ ]:
def stable_inverse_or_pinv(A: np.ndarray) -> np.ndarray:
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A)

def ridge_predict(X: np.ndarray, y: np.ndarray, lambdas: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X.T @ X + np.diag(lambdas)
    b = X.T @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b
    return beta, X @ beta

def build_eval_grid_from_data(rx: Sequence[float], ry: Sequence[float], grid_n: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    rx_min, rx_max = np.nanmin(rx), np.nanmax(rx)
    ry_min, ry_max = np.nanmin(ry), np.nanmax(ry)
    if not np.isfinite(rx_min) or not np.isfinite(rx_max):
        rx_min, rx_max = -1.0, 1.0
    if not np.isfinite(ry_min) or not np.isfinite(ry_max):
        ry_min, ry_max = -1.0, 1.0
    if rx_min == rx_max:
        rx_min -= 1.0
        rx_max += 1.0
    if ry_min == ry_max:
        ry_min -= 1.0
        ry_max += 1.0
    gx = np.linspace(rx_min, rx_max, grid_n)
    gy = np.linspace(ry_min, ry_max, grid_n)
    RX, RY = np.meshgrid(gx, gy)
    return RX.ravel(), RY.ravel()

def gopt_value(
    X_meas: np.ndarray,
    X_eval: np.ndarray,
    lambdas: np.ndarray,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
) -> float:
    """
    VBA G_optimal_type 반영.
    - Type1: H H.T diag의 sqrt(max)
    - Type2: H matrix element의 signed max의 sqrt. 음수이면 999.0 반환.
    - Type3: X_eval inv(...) X_eval.T diag의 sqrt(max)
    - Type4: inner=Type2, partial=Type1
    """
    X_meas = np.asarray(X_meas, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X_meas.T @ X_meas + np.diag(lambdas)
    invA = stable_inverse_or_pinv(A)
    effective_type = gopt_type
    if gopt_type == 4:
        effective_type = 2 if inner_or_partial == "inner" else 1
    if effective_type == 1:
        H = X_eval @ invA @ X_meas.T
        diag_val = np.sum(H * H, axis=1)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    if effective_type == 2:
        H = X_eval @ invA @ X_meas.T
        max_val = float(np.nanmax(H))
        if max_val < 0:
            return 999.0
        return float(np.sqrt(max_val))
    if effective_type == 3:
        M = X_eval @ invA @ X_eval.T
        diag_val = np.diag(M)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    raise ValueError(f"Unsupported gopt_type: {gopt_type}")


## 06. Span/box 제약 및 한 방향 회귀 solver
원본 MRC_PSM_decorrect의 25번 정의를 그대로 포함합니다. 실제 설정은 07에서 저장 설정 및 00의 override를 반영합니다.


In [ ]:
def calc_contribution_spans(X_eval: np.ndarray, beta: np.ndarray, coeff_names: Sequence[str]) -> Dict[str, float]:
    spans = {}
    for i, name in enumerate(coeff_names):
        contrib = X_eval[:, i] * beta[i]
        spans[name] = float(np.nanmax(contrib) - np.nanmin(contrib))
    return spans

def find_span_violations(
    spans: Dict[str, float],
    span_limits: Dict[str, Dict[str, float]],
    tolerance: float = 0.01,
) -> List[Tuple[str, float, float, float]]:
    violations = []
    for name, span in spans.items():
        if name not in span_limits:
            continue
        lo = span_limits[name]["min"] - tolerance
        hi = span_limits[name]["max"] + tolerance
        if span < lo or span > hi:
            violations.append((name, span, span_limits[name]["min"], span_limits[name]["max"]))
    return violations

def build_beta_bounds_from_span(
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA의 min_Span_X/Y, max_Span_X/Y에 대응하는 box constraint 근사.
    contribution span = range(X_j * beta_j)가 span_limit 안에 들어오도록
    beta_j bound를 계산한다.
    """
    n = len(coeff_names)
    lo = np.full(n, -np.inf, dtype=float)
    hi = np.full(n, np.inf, dtype=float)
    for j, name in enumerate(coeff_names):
        if name not in span_limits:
            continue
        lim = span_limits[name]
        span_abs = max(abs(float(lim["min"])), abs(float(lim["max"])))
        xj_range = float(np.nanmax(X_eval[:, j]) - np.nanmin(X_eval[:, j]))
        if xj_range <= 0 or not np.isfinite(xj_range):
            continue
        b = span_abs / xj_range
        lo[j] = -b
        hi[j] = b
    return lo, hi

def projected_coordinate_descent_ridge(
    X: np.ndarray,
    y: np.ndarray,
    lambdas: np.ndarray,
    beta_init: Optional[np.ndarray] = None,
    lower: Optional[np.ndarray] = None,
    upper: Optional[np.ndarray] = None,
    tol: float = 1e-6,
    max_iter: int = 5000,
) -> Tuple[np.ndarray, np.ndarray, int, float]:
    """
    VBA ProjectedCoordinateDescentElasticNet(l1=0) 근사.
    solve min ||y-Xb||^2 + sum(lambda_j*b_j^2), s.t. lower <= b <= upper
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    n_param = X.shape[1]
    if beta_init is None:
        beta, _ = ridge_predict(X, y, lambdas)
    else:
        beta = np.asarray(beta_init, dtype=float).copy()
    if lower is None:
        lower = np.full(n_param, -np.inf)
    if upper is None:
        upper = np.full(n_param, np.inf)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    beta = np.clip(beta, lower, upper)
    residual = y - X @ beta
    col_norm2 = np.sum(X * X, axis=0)
    last_delta = np.inf
    for it in range(1, max_iter + 1):
        max_delta = 0.0
        for j in range(n_param):
            denom = col_norm2[j] + lambdas[j]
            if denom <= 0 or not np.isfinite(denom):
                continue
            residual += X[:, j] * beta[j]
            rho = float(X[:, j] @ residual)
            new_beta = rho / denom
            new_beta = min(max(new_beta, lower[j]), upper[j])
            delta = abs(new_beta - beta[j])
            beta[j] = new_beta
            residual -= X[:, j] * beta[j]
            if delta > max_delta:
                max_delta = delta
        last_delta = max_delta
        if max_delta < tol:
            break
    return beta, X @ beta, it, float(last_delta)

@dataclass
class RidgeFitResult:
    beta: np.ndarray
    pred: np.ndarray
    lambdas: np.ndarray
    gopt: float
    spans: Dict[str, float]
    violations: List[Tuple[str, float, float, float]]
    gopt_iter: int
    span_iter: int
    method: str

def solve_vba_style_ridge_one_axis(
    X_fit: np.ndarray,
    y: np.ndarray,
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
    ridge_regression: bool = True,
    gopt_spec: float = 1.0,
    gopt_tolerance: float = 0.001,
    minimum_ridge: float = 0.001,
    add_ridge: float = 10.0,
    max_gopt_return: int = 50,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
    use_span_box_constraint: bool = True,
    span_tolerance: float = 0.01,
    cd_tol: float = 1e-6,
    max_span_iter: int = 5000,
    base_lambda_scale: float = 1.0,
    debug: bool = False,
) -> RidgeFitResult:
    """
    VBA flow에 맞춘 한 방향(X or Y) fitting.
    - 초기 ridge는 0에 가까운 값이 아니라 VBA minimum_Ridge 기준 사용
    - Gopt가 spec 이상이면 ridge를 빠르게 키운 뒤 이분탐색으로 spec 근처를 찾음
    - span over가 남으면 projected coordinate descent로 coefficient를 box constraint 안에 넣음
    """
    X_fit = np.asarray(X_fit, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    y = np.asarray(y, dtype=float)
    n_param = X_fit.shape[1]
    if n_param != len(coeff_names):
        raise ValueError(f"n_param({n_param}) != len(coeff_names)({len(coeff_names)})")
    initial_lambda = 0.0 if not ridge_regression else 0.0
    lambdas = make_lambda_vector(initial_lambda, coeff_names)
    beta, pred = ridge_predict(X_fit, y, lambdas)
    g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
    gopt_iter = 0
    method = "ols"
    if ridge_regression and (not np.isfinite(g) or g >= gopt_spec or g == 0):
        low_lambda = 0.0
        high_lambda = max(minimum_ridge * base_lambda_scale, minimum_ridge)
        high_lambda = max(high_lambda, minimum_ridge)
        high_lambdas = make_lambda_vector(high_lambda, coeff_names)
        high_g = gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        while (not np.isfinite(high_g) or high_g >= gopt_spec) and gopt_iter < max_gopt_return:
            low_lambda = high_lambda
            high_lambda = high_lambda * add_ridge
            high_lambdas = make_lambda_vector(high_lambda, coeff_names)
            high_g = gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
            gopt_iter += 1
            if debug:
                print(f"fast ridge: lambda={high_lambda:.6g}, gopt={high_g:.6g}")
        target_low = max(0.0, gopt_spec - gopt_tolerance)
        target_high = gopt_spec
        best_lambda = high_lambda
        best_g = high_g
        if np.isfinite(high_g) and high_g < gopt_spec:
            lo = low_lambda
            hi = high_lambda
            for _ in range(max_gopt_return):
                mid = (lo + hi) / 2.0
                mid_lambdas = make_lambda_vector(mid, coeff_names)
                mid_g = gopt_value(X_fit, X_eval, mid_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
                gopt_iter += 1
                if debug:
                    print(f"slow ridge: lambda={mid:.6g}, gopt={mid_g:.6g}")
                if target_low <= mid_g < target_high:
                    best_lambda = mid
                    best_g = mid_g
                    break
                if mid_g >= gopt_spec or not np.isfinite(mid_g):
                    lo = mid
                else:
                    hi = mid
                    best_lambda = mid
                    best_g = mid_g
                if abs(hi - lo) < max(minimum_ridge * 1e-3, 1e-12):
                    break
        lambdas = make_lambda_vector(best_lambda, coeff_names)
        beta, pred = ridge_predict(X_fit, y, lambdas)
        g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt"
    spans = calc_contribution_spans(X_eval, beta, coeff_names)
    violations = find_span_violations(spans, span_limits, tolerance=span_tolerance)
    span_iter = 0
    if ridge_regression and use_span_box_constraint and len(violations) > 0:
        lower, upper = build_beta_bounds_from_span(X_eval, coeff_names, span_limits)
        beta_cd, pred_cd, span_iter, _ = projected_coordinate_descent_ridge(
            X_fit,
            y,
            lambdas,
            beta_init=beta,
            lower=lower,
            upper=upper,
            tol=cd_tol,
            max_iter=max_span_iter,
        )
        beta = beta_cd
        pred = pred_cd
        spans = calc_contribution_spans(X_eval, beta, coeff_names)
        violations = find_span_violations(spans, span_limits, tolerance=span_tolerance)
        g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt_box_cd"
    return RidgeFitResult(
        beta=beta,
        pred=pred,
        lambdas=lambdas,
        gopt=float(g),
        spans=spans,
        violations=violations,
        gopt_iter=int(gopt_iter),
        span_iter=int(span_iter),
        method=method,
    )

def determine_inner_or_partial(n_valid: int, n_virtual: int) -> str:
    """VBA: virtual_points = Vgrid_XY면 inner, 아니면 partial."""
    return "inner" if n_valid >= n_virtual else "partial"


## 07. 회전 설정 / 모델 대응표 / X 한 방향 solver


In [ ]:
# 03~06의 원본 solver 정의를 먼저 실행합니다.
SETTINGS_MAP = {
    "span_mode": "SPAN_MODE", "min_points_guide": "MIN_POINTS_GUIDE", "eval_grid_n": "CPE_EVAL_GRID_N", "ridge_regression": "RIDGE_REGRESSION",
    "gopt_spec": "G_OPT_SPEC", "gopt_tolerance": "G_OPT_TOLERANCE", "gopt_type": "G_OPTIMAL_TYPE", "use_span_box_constraint": "USE_SPAN_BOX_CONSTRAINT",
    "use_virtual_chip_points": "USE_VIRTUAL_CHIP_POINTS", "virtual_wafer_radius_um": "VIRTUAL_WAFER_RADIUS", "span_tolerance": "SPAN_TOLERANCE",
    "minimum_ridge": "MINIMUM_RIDGE", "add_ridge": "ADD_RIDGE", "max_gopt_return": "MAX_GOPT_RETURN", "max_span_iter": "MAX_SPAN_ITER", "cd_tol": "CD_TOL", "ridge_valves": "RIDGE_VALVE_HYPER",
}
settings_path = STAGE_DIR.parent / "extension_settings.json"
saved_settings = json.loads(settings_path.read_text(encoding="utf-8")) if USE_SAVED_SOLVER_SETTINGS and settings_path.is_file() else {}
for source, value in {**saved_settings, **saved_settings.get("cpe_options", {})}.items():
    if source in SETTINGS_MAP:
        globals()[SETTINGS_MAP[source]] = value
NATIVE_SPAN_LIMITS = saved_settings.get("active_span_limits", get_active_span_limits(SPAN_MODE))
for key, value in SOLVER_OVERRIDES.items():
    if key not in set(SETTINGS_MAP.values()) | {"NATIVE_SPAN_LIMITS"}:
        raise ValueError(f"SOLVER_OVERRIDES: 지원하지 않는 설정 {key}")
    globals()[key] = value
if "SPAN_MODE" in SOLVER_OVERRIDES and "NATIVE_SPAN_LIMITS" not in SOLVER_OVERRIDES:
    NATIVE_SPAN_LIMITS = get_active_span_limits(SPAN_MODE)
ROTATION_SIGN = 1 if ROTATION_DEG == 90 else -1
X_POWERS = [(0,0), (1,0), (0,1), (2,0), (1,1), (0,2), (3,0), (2,1), (1,2), (0,3), (3,1), (2,2), (1,3), (0,4), (3,2), (2,3), (1,4), (0,5), (3,3), (2,4), (3,4)]
Y_POWERS = [(0,0), (0,1), (1,0), (0,2), (1,1), (2,0), (0,3), (1,2), (2,1), (0,4), (1,3), (2,2), (0,5), (1,4), (2,3), (1,5), (2,4)]
DEGREE_SCALE = {0: 1., 1: 1e6, 2: 1e9, 3: 1e12, 4: 1e19, 5: 1e23, 6: 1e27, 7: 1e31}

def rotate_local(rx, ry):
    return -ROTATION_SIGN * np.asarray(ry), ROTATION_SIGN * np.asarray(rx)

def model_table():
    rows = []
    original_set, rotated_set = set(X_POWERS), {(b,a) for a,b in Y_POWERS}
    for case, names, powers in [(NATIVE_CASES[0], CPE38_KEYS_DX, X_POWERS), (NATIVE_CASES[1], CPE38_KEYS_DY, Y_POWERS)]:
        for name, (a,b) in zip(names, powers):
            rotated = case == NATIVE_CASES[1]
            px, py = (b,a) if rotated else (a,b)
            sign = ROTATION_SIGN * (-ROTATION_SIGN)**a * ROTATION_SIGN**b if rotated else (-1 if name == "RK5" else 1)
            rows.append(dict(
                case=case, term=name, power_x=px, power_y=py, sign_to_original_x=sign, scale=DEGREE_SCALE[a+b], shared_monomial=(px,py) in original_set & rotated_set,
                ridge_valve=get_ridge_valves([name])[0], span_limit_min=NATIVE_SPAN_LIMITS.get(name, {}).get("min", np.nan), span_limit_max=NATIVE_SPAN_LIMITS.get(name, {}).get("max", np.nan),
            ))
    return pd.DataFrame(rows)

def design_information(matrix):
    norms = np.linalg.norm(matrix, axis=0)
    scaled = matrix / np.where(norms > 0, norms, 1.)
    singular = np.linalg.svd(scaled, compute_uv=False)
    rank = int(np.linalg.matrix_rank(scaled))
    condition = float(singular[0] / singular[-1]) if rank == matrix.shape[1] else np.inf
    return rank, condition

def solve_case(matrix, target, evaluation, names, n_virtual, diagnostic=False):
    if diagnostic:
        norms = np.linalg.norm(matrix, axis=0)
        norms = np.where(norms > 0, norms, 1.)
        beta = np.linalg.lstsq(matrix / norms, target, rcond=None)[0] / norms
        return RidgeFitResult(beta, matrix @ beta, np.zeros(len(names)), np.nan, calc_contribution_spans(evaluation, beta, names), [], 0, 0, "ols_diagnostic_column_scaled")
    return solve_vba_style_ridge_one_axis(
        matrix, target, evaluation, names, NATIVE_SPAN_LIMITS, ridge_regression=RIDGE_REGRESSION, gopt_spec=G_OPT_SPEC, gopt_tolerance=G_OPT_TOLERANCE,
        minimum_ridge=MINIMUM_RIDGE, add_ridge=ADD_RIDGE, max_gopt_return=MAX_GOPT_RETURN, gopt_type=G_OPTIMAL_TYPE,
        inner_or_partial=determine_inner_or_partial(len(target), n_virtual), use_span_box_constraint=USE_SPAN_BOX_CONSTRAINT,
        span_tolerance=SPAN_TOLERANCE, cd_tol=CD_TOL, max_span_iter=MAX_SPAN_ITER, base_lambda_scale=choose_base_lambda_scale(len(target)),
    )

term_mapping = model_table()
save_table("07_TERM_MAPPING", term_mapping)
display(term_mapping)
print("공통 단항식:", int(term_mapping.loc[term_mapping["case"].eq(NATIVE_CASES[0]), "shared_monomial"].sum()), "/ X21과 회전 Y17은 포함관계가 아닙니다.")
print("설정:", "저장 설정 반영" if saved_settings else "원본 노트북 기본값", "; 회전:", ROTATION_DEG)


## 08. 동일 X point에서 X21 / 회전 Y17 shot별 회귀 및 저장


In [ ]:
def fit_rotation_mode(frame, mode):
    out = frame.copy()
    cases = NATIVE_CASES + (["X21_OLS_DIAGNOSTIC", "ROT_Y17_OLS_DIAGNOSTIC"] if RUN_OLS_DIAGNOSTIC else [])
    for case in cases:
        out[f"{case}_fit_x"] = np.nan
        out[f"{case}_residual_x"] = np.nan
    coefficients, diagnostics, reference_norms = [], [], {}
    mapping_lookup = {case: term_mapping.loc[term_mapping["case"].eq(case)].set_index("term").to_dict("index") for case in NATIVE_CASES}
    # wafer 내 shot 간 term 산포를 비교하므로 nm 환산 좌표 범위를 wafer별로 고정합니다.
    for wafer_key, work in out.loc[out["rotation_valid_x"]].groupby(WAFER_KEYS, sort=False):
        coordinates = work[COORDS].drop_duplicates().to_numpy(float)
        reference_x, _ = cpe_38para(coordinates[:,0], coordinates[:,1])
        u, v = rotate_local(coordinates[:,0], coordinates[:,1])
        _, reference_y = cpe_38para(u, v)
        reference_norms[wafer_key] = {NATIVE_CASES[0]: np.max(np.abs(reference_x), axis=0), NATIVE_CASES[1]: np.max(np.abs(reference_y), axis=0)}
    grouped = out.loc[out["rotation_valid_x"]].groupby(SHOT_KEYS, dropna=False, sort=False)
    for shot_index, (key, work) in enumerate(grouped):
        shot = dict(zip(SHOT_KEYS, key))
        rx, ry, error_x = (work[col].to_numpy(float) for col in COORDS + [TARGET])
        if USE_VIRTUAL_CHIP_POINTS:
            gx, gy, virtual_source, n_virtual_full, n_virtual = get_virtual_points_for_group(work, rx, ry, float(shot["fcp_x"]), float(shot["fcp_y"]), wafer_radius=VIRTUAL_WAFER_RADIUS)
        else:
            gx, gy = build_eval_grid_from_data(rx, ry, grid_n=CPE_EVAL_GRID_N)
            virtual_source, n_virtual_full, n_virtual = "measured_range_grid", len(gx), len(gx)
        # 원래 wafer 경계로 자른 virtual point를 회전합니다. 미회전 shot 중심으로 다시 자르지 않습니다.
        ux, uy = rotate_local(rx, ry)
        gux, guy = rotate_local(gx, gy)
        original_matrix, _ = cpe_38para(rx, ry)
        original_eval, _ = cpe_38para(gx, gy)
        _, rotated_matrix = cpe_38para(ux, uy)
        _, rotated_eval = cpe_38para(gux, guy)
        ranks = {False: design_information(original_matrix), True: design_information(rotated_matrix)}
        for case in cases:
            rotated = case.startswith("ROT_")
            matrix, evaluation = (rotated_matrix, rotated_eval) if rotated else (original_matrix, original_eval)
            names = CPE38_KEYS_DY if rotated else CPE38_KEYS_DX
            target, back_sign = (ROTATION_SIGN * error_x, ROTATION_SIGN) if rotated else (error_x, 1)
            fit = solve_case(matrix, target, evaluation, names, n_virtual, diagnostic=case.endswith("DIAGNOSTIC"))
            prediction = back_sign * fit.pred
            if not np.isfinite(prediction).all() or not np.isfinite(fit.beta).all():
                raise ValueError(f"회귀 결과에 비유한 값 발생: {mode}, {shot}, {case}")
            out.loc[work.index, f"{case}_fit_x"] = prediction
            out.loc[work.index, f"{case}_residual_x"] = error_x - prediction
            rank, condition = ranks[rotated]
            diag = dict(
                mode=mode, **shot, case=case, n_valid_x=len(work), n_parameters=len(names), rank=rank, full_column_rank=rank == len(names), column_scaled_condition=condition,
                n_virtual=n_virtual, n_virtual_full=n_virtual_full, virtual_source=virtual_source, inner_or_partial=determine_inner_or_partial(len(work), n_virtual),
                method=fit.method, gopt=fit.gopt, lambda_max=float(np.max(fit.lambdas)), gopt_iterations=fit.gopt_iter, span_iterations=fit.span_iter,
                span_violation_count=len(fit.violations), span_violation_terms="|".join(item[0] for item in fit.violations),
                gopt_within_spec=bool(np.isfinite(fit.gopt) and 0 < fit.gopt < G_OPT_SPEC), max_decomposition_error=float(np.max(np.abs(error_x - (prediction + (error_x - prediction))))),
            )
            diagnostics.append(diag)
            coef = dict(mode=mode, **shot, case=case, coefficient_coordinates="rotated_local_Y" if rotated else "original_local_X", rotation_deg=ROTATION_DEG if rotated else 0)
            native_case = NATIVE_CASES[1] if rotated else NATIVE_CASES[0]
            mapping = mapping_lookup[native_case]
            max_abs_reference = reference_norms[tuple(shot[col] for col in WAFER_KEYS)][native_case]
            for index, name in enumerate(names):
                mono_sign = int(mapping[name]["sign_to_original_x"])
                coef[f"beta_{name}"] = float(fit.beta[index])
                coef[f"signed_peak_nm_{name}"] = float(fit.beta[index] * mono_sign * max_abs_reference[index] * UM_TO_NM)
                coef[f"reference_max_abs_basis_{name}"] = float(max_abs_reference[index])
                coef[f"span_nm_{name}"] = float(fit.spans[name] * UM_TO_NM)
                coef[f"lambda_{name}"] = float(fit.lambdas[index])
            coefficients.append(coef)
        if (shot_index + 1) % 250 == 0:
            print(f"{mode}: {shot_index + 1:,} shots 완료")
    diag_frame = pd.DataFrame(diagnostics)
    if diag_frame.empty:
        raise ValueError(f"{mode}: 회귀 가능한 X point가 없습니다.")
    return out, pd.DataFrame(coefficients), diag_frame

results, all_coefficients, all_diagnostics = {}, [], []
for mode in MODES:
    points, coefficients, diagnostics = fit_rotation_mode(frames[mode], mode)
    results[mode] = points
    all_coefficients.append(coefficients)
    all_diagnostics.append(diagnostics)
    save_table(f"08_{mode}_ROTATION_POINTS", points, excel=False)
    save_table(f"08_{mode}_SHOT_COEFFICIENTS", coefficients)
    save_table(f"08_{mode}_SHOT_DIAGNOSTICS", diagnostics)
coefficient_table = pd.concat(all_coefficients, ignore_index=True)
diagnostic_table = pd.concat(all_diagnostics, ignore_index=True)
save_table("08_ALL_SHOT_DIAGNOSTICS", diagnostic_table)
display(diagnostic_table.head(PREVIEW_ROWS))


## 09. Wafer/shot 통계 / CPE budget / term별 산포 비교


In [ ]:
def scalar_statistics(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)] * UM_TO_NM
    n = len(values)
    mean = float(np.mean(values)) if n else np.nan
    sigma = float(np.std(values, ddof=STD_DDOF)) if n > STD_DDOF else np.nan
    return dict(n=n, mean_nm=mean, std_nm=sigma, three_sigma_nm=3*sigma, mean_plus_3sigma_nm=mean+3*sigma, abs_mean_plus_3sigma_nm=abs(mean)+3*sigma, rms_nm=float(np.sqrt(np.mean(values**2))) if n else np.nan)

def summarize_points(points, mode, group_keys):
    rows = []
    for key, work in points.groupby(group_keys, dropna=False, sort=False):
        identity = dict(zip(group_keys, key))
        valid = work.loc[work["rotation_valid_x"]]
        for case in NATIVE_CASES + (["X21_OLS_DIAGNOSTIC", "ROT_Y17_OLS_DIAGNOSTIC"] if RUN_OLS_DIAGNOSTIC else []):
            row = dict(mode=mode, **identity, case=case, n_total=len(work), n_invalid_x=len(work)-len(valid))
            for component, col in [("input", TARGET), ("cpe", f"{case}_fit_x"), ("residual", f"{case}_residual_x")]:
                row.update({f"{component}_{name}": value for name,value in scalar_statistics(valid[col]).items()})
            if len(valid) > STD_DDOF:
                covariance = float(np.cov(valid[f"{case}_fit_x"], valid[f"{case}_residual_x"], ddof=STD_DDOF)[0,1] * UM_TO_NM**2)
                row["cpe_residual_covariance_nm2"] = covariance
                row["variance_budget_error_nm2"] = row["input_std_nm"]**2 - row["cpe_std_nm"]**2 - row["residual_std_nm"]**2 - 2*covariance
            rows.append(row)
    return pd.DataFrame(rows)

def paired_comparison(statistics, group_keys):
    left = statistics.loc[statistics["case"].eq(NATIVE_CASES[0])].drop(columns="case")
    right = statistics.loc[statistics["case"].eq(NATIVE_CASES[1])].drop(columns="case")
    pair = left.merge(right, on=["mode"]+group_keys, how="outer", validate="one_to_one", suffixes=("_X21", "_ROT_Y17"), indicator=True)
    if not pair["_merge"].eq("both").all():
        raise ValueError("X21/Y17 비교 키가 일치하지 않습니다.")
    pair = pair.drop(columns="_merge")
    for metric in ["mean_nm", "three_sigma_nm", "abs_mean_plus_3sigma_nm", "rms_nm"]:
        for component in ["cpe", "residual"]:
            prefix = f"{component}_{metric}"
            pair[f"{prefix}_delta_ROT_minus_X21"] = pair[f"{prefix}_ROT_Y17"] - pair[f"{prefix}_X21"]
    pair["residual_3sigma_reduction_pct"] = 100 * (1 - pair["residual_three_sigma_nm_ROT_Y17"] / pair["residual_three_sigma_nm_X21"].where(pair["residual_three_sigma_nm_X21"] > 1e-12))
    return pair

wafer_statistics = pd.concat([summarize_points(points, mode, WAFER_KEYS) for mode,points in results.items()], ignore_index=True)
shot_statistics = pd.concat([summarize_points(points, mode, SHOT_KEYS) for mode,points in results.items()], ignore_index=True)
wafer_comparison, shot_comparison = paired_comparison(wafer_statistics, WAFER_KEYS), paired_comparison(shot_statistics, SHOT_KEYS)
term_rows = []
for (mode, case), coefficients in coefficient_table.groupby(["mode", "case"], sort=False):
    names = CPE38_KEYS_DY if case.startswith("ROT_") else CPE38_KEYS_DX
    for key, group in coefficients.groupby(WAFER_KEYS, dropna=False, sort=False):
        for name in names:
            peaks = group[f"signed_peak_nm_{name}"]
            term_rows.append(dict(mode=mode, **dict(zip(WAFER_KEYS,key)), case=case, term=name, n_shots=len(group), peak_mean_nm=float(peaks.mean()), peak_3sigma_nm=float(3*peaks.std(ddof=STD_DDOF)), mean_abs_peak_nm=float(peaks.abs().mean())))
term_wafer_statistics = pd.DataFrame(term_rows)
for name, table in [("09_WAFER_STATISTICS", wafer_statistics), ("09_SHOT_STATISTICS", shot_statistics), ("09_WAFER_COMPARISON", wafer_comparison), ("09_SHOT_COMPARISON", shot_comparison), ("09_TERM_WAFER_STATISTICS", term_wafer_statistics)]:
    save_table(name, table)
display(wafer_comparison[["mode"] + WAFER_KEYS + ["residual_three_sigma_nm_X21", "residual_three_sigma_nm_ROT_Y17", "residual_3sigma_reduction_pct"]].head(PREVIEW_ROWS))


## 10. Wafer별 대응 비교 그래프


In [ ]:
plot_paths = []
for mode in MODES:
    pair = wafer_comparison.loc[wafer_comparison["mode"].eq(mode)].reset_index(drop=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.7))
    for ax, (component, metric, title) in zip(axes, [("cpe", "three_sigma_nm", "Removed CPE 3sigma"), ("residual", "three_sigma_nm", "Final residual 3sigma"), ("residual", "abs_mean_plus_3sigma_nm", "Final residual |mean|+3sigma")]):
        x, y = pair[f"{component}_{metric}_X21"].to_numpy(float), pair[f"{component}_{metric}_ROT_Y17"].to_numpy(float)
        valid = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[valid], y[valid], color="tab:blue", s=23, alpha=.7)
        limit = max(float(np.max(np.r_[x[valid],y[valid]])) * 1.06, 1e-6) if valid.any() else 1.
        ax.plot([0,limit], [0,limit], ":", color="gray", label="Equal")
        ax.set(xlabel="Original X21 [nm]", ylabel="Rotated Y17, restored X [nm]", title=title, xlim=(0,limit), ylim=(0,limit))
        ax.text(.03,.97,f"N={int(valid.sum())}; one point = one wafer", transform=ax.transAxes, va="top", fontsize=9)
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    fig.suptitle(f"{mode}: same critical-X points / rotation {ROTATION_DEG:+d} deg")
    fig.tight_layout()
    path = PLOT_DIR / f"{mode}_X21_vs_rotated_Y17.png"
    fig.savefig(path, dpi=170, bbox_inches="tight")
    fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    plot_paths.append(str(path))
    if SHOW_PLOTS:
        display(fig)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(max(9, min(20, len(pair)*.13)), 4.5))
    indices = np.arange(len(pair))
    ax.plot(indices, pair["residual_three_sigma_nm_X21"], "o-", ms=3, lw=.7, label="Original X21")
    ax.plot(indices, pair["residual_three_sigma_nm_ROT_Y17"], "o-", ms=3, lw=.7, label="Rotated Y17 -> original X")
    ax.set(xlabel="Wafer index (see 10_WAFER_PLOT_INDEX)", ylabel="Final residual 3sigma [nm]", title=f"{mode}: paired wafer residual comparison")
    ax.grid(alpha=.2)
    ax.legend()
    fig.tight_layout()
    path = PLOT_DIR / f"{mode}_wafer_residual_3sigma.png"
    fig.savefig(path, dpi=170, bbox_inches="tight")
    plot_paths.append(str(path))
    if SHOW_PLOTS:
        display(fig)
    plt.close(fig)
plot_index = pd.concat([wafer_comparison.loc[wafer_comparison["mode"].eq(mode), ["mode"]+WAFER_KEYS].reset_index(drop=True).rename_axis("wafer_index").reset_index() for mode in MODES], ignore_index=True)
save_table("10_WAFER_PLOT_INDEX", plot_index)


## 11. 비교 Excel / 설정 / 실행 manifest 저장


In [ ]:
notes = pd.DataFrame([
    ("input", f"{TARGET}: raw에서 wafer WK/RK를 제거한 X. WK/RK는 재계산하지 않습니다."),
    ("critical_axis", "두 모델 모두 원래 장비 좌표의 X만 비교합니다."),
    ("rotation", "u=-s*y, v=s*x, error_v=s*error_x; fit_x=s*fit_v; s=+1(+90), -1(-90)."),
    ("X21_NATIVE", "원본 X21의 ridge/G-opt/span 설정으로 재회귀."),
    ("ROT_Y17_NATIVE", "회전한 좌표와 X 오차를 Y17로 회귀하고 원래 X로 복원. Y측 ridge/span 사용."),
    ("OLS_DIAGNOSTIC", "같은 point의 제약 없는 회귀. 학습 point 설명력 진단이며 예측 성능을 뜻하지 않습니다."),
    ("term_peak", "원래 X 단항식으로 변환한 계수 × max(abs(basis)) ×1000. 같은 wafer의 전체 유효 측정 좌표를 공통 기준으로 사용."),
    ("budget", "Var(input)=Var(CPE)+Var(residual)+2Cov(CPE,residual). 3sigma나 variance 단순 가산 불가."),
    ("interpretation", "제거 CPE가 작다는 것만으로 개선으로 판단하지 않습니다. 최종 residual을 함께 저장. 장비 보정은 적용하지 않습니다."),
    ("wafer_statistics", "wafer의 전체 유효 point에서 재계산. shot별 3sigma 평균이 아닙니다."),
    ("product_filter", "FILTERS가 비면 전체 입력을 critical-X 제품으로 취급. 정상 critical-Y 제품을 섞지 마세요."),
    ("missing_Y", "Y 실측값이나 Y residual 불필요. X와 좌표가 유한한 동일 point로 두 모델 비교."),
], columns=["item", "description"])
report_path = OUTPUT_DIR / "CPE_Critical_Rotation_Comparison.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    for sheet, table in [("Wafer_comparison", wafer_comparison), ("Shot_comparison", shot_comparison), ("Wafer_statistics", wafer_statistics), ("Term_wafer_statistics", term_wafer_statistics), ("Shot_diagnostics", diagnostic_table), ("Term_mapping", term_mapping), ("Selected_wafers", selection), ("Notes", notes)]:
        if len(table) > 1048575:
            raise ValueError(f"{sheet}: Excel 행 수 제한 초과. tables의 전체 Parquet/CSV를 사용하세요.")
        excel_safe(table).to_excel(writer, sheet_name=sheet, index=False)
        ws = writer.book[sheet]
        ws.freeze_panes = "E2"
        ws.auto_filter.ref = ws.dimensions
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(42, max(16, len(str(col[0].value))+2))
actual_settings = {name: globals()[name] for name in sorted(set(SETTINGS_MAP.values()))}
actual_settings["active_span_limits"] = NATIVE_SPAN_LIMITS
run_settings = dict(
    modes=MODES, filters=FILTERS, filters_by_mode=FILTERS_BY_MODE, rotation_deg=ROTATION_DEG, target=TARGET, input_units="um", output_units="nm",
    std_ddof=STD_DDOF, ols_diagnostic=RUN_OLS_DIAGNOSTIC, term_nm_reference="wafer_measured_points_common_across_shots",
    saved_settings_path=str(settings_path) if saved_settings else None, solver=actual_settings, solver_overrides=SOLVER_OVERRIDES, plots=plot_paths,
)
(OUTPUT_DIR / "rotation_settings.json").write_text(json.dumps(run_settings, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(input_manifest).to_csv(OUTPUT_DIR / "input_manifest.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(output_manifest.values()).to_csv(OUTPUT_DIR / "saved_files.csv", index=False, encoding="utf-8-sig")
display(FileLink(os.path.relpath(report_path, Path.cwd())))
print("완료: 같은 critical X 데이터의 X21 / 회전 Y17 비교. 기존 stage와 원본 노트북은 변경하지 않았습니다.")
